# Лабораторная 5 - Обработка изображений в OpenCV

ФИО, группа:

## 1. Библиотеки и файлы

In [ ]:
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display

plt.rcParams.update({"figure.figsize": (10, 5), "axes.unicode_minus": False})
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

In [ ]:
PHOTO_PATH = Path("mydata/photo.jpg")
BASE_PATH = PHOTO_PATH  # Можно указать другую фотографию для наложения PNG
GROUP_PATH = Path("mydata/group.jpg")
OVERLAY_PATH = Path("mydata/overlay.png")
VIDEO_PATH = Path("mydata/video.mp4")
BACKGROUND_PATH = Path("mydata/background.jpg")
FRAME_INDEX = 0  # Номер кадра для подбора цвета фона, нумерация с нуля

In [ ]:
def read_image(path, unchanged=False):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"Файл не найден: {path}")
    flag = cv2.IMREAD_UNCHANGED if unchanged else cv2.IMREAD_COLOR
    image = cv2.imread(str(path), flag)
    if image is None:
        raise ValueError(f"OpenCV не смог прочитать изображение: {path}")
    return image


def show_bgr(ax, image, title):
    ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    ax.set_title(title)
    ax.set_axis_off()


photo = read_image(PHOTO_PATH)
height, width, channels = photo.shape
print("Форма массива (высота, ширина, каналы):", photo.shape)
print("Тип элементов:", photo.dtype)
print("Число пикселей:", height * width)
print("Число элементов массива:", photo.size)
print("Пиксель в левом верхнем углу, BGR:", photo[0, 0])

## 2. Изображение и каналы

Первый вывод использует cv2_imshow в Colab или Pillow в Jupyter. Второй - Matplotlib с преобразованием BGR в RGB

In [ ]:
try:
    from google.colab.patches import cv2_imshow
except ImportError:
    display(Image.fromarray(cv2.cvtColor(photo, cv2.COLOR_BGR2RGB)))
else:
    cv2_imshow(photo)

fig, ax = plt.subplots()
show_bgr(ax, photo, "Исходное изображение")
plt.show()

In [ ]:
blue, green, red = cv2.split(photo)
gray = cv2.cvtColor(photo, cv2.COLOR_BGR2GRAY)
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, channel, title in zip(axes, [blue, green, red, gray],
                             ["B", "G", "R", "Оттенки серого"]):
    ax.imshow(channel, cmap="gray", vmin=0, vmax=255)
    ax.set_title(title)
    ax.set_axis_off()
plt.show()

Наблюдения: различия между каналами, отличие серого изображения от отдельного канала, смысл значений массива

## 3. Поиск лиц

In [ ]:
group = read_image(GROUP_PATH)
group_gray = cv2.cvtColor(group, cv2.COLOR_BGR2GRAY)
cascade_path = Path(cv2.data.haarcascades) / "haarcascade_frontalface_default.xml"
detector = cv2.CascadeClassifier(str(cascade_path))
if detector.empty():
    raise RuntimeError(f"Не удалось загрузить каскад: {cascade_path}")

FACE_SETTINGS = [(1.05, 3), (1.10, 5), (1.20, 7)]
MIN_FACE_SIZE = (30, 30)
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, (scale_factor, neighbors) in zip(axes, FACE_SETTINGS):
    faces = detector.detectMultiScale(
        group_gray, scaleFactor=scale_factor,
        minNeighbors=neighbors, minSize=MIN_FACE_SIZE,
    )
    marked = group.copy()
    for x, y, w, h in faces:
        cv2.rectangle(marked, (x, y), (x + w, y + h), (0, 255, 0), 2)
    show_bgr(ax, marked, f"scaleFactor={scale_factor}, minNeighbors={neighbors}\nОбнаружений: {len(faces)}")
plt.show()

Число видимых лиц на исходной фотографии:

Сравнение настроек: правильные обнаружения, пропущенные лица и ложные обнаружения

Выбранные параметры и причина выбора:

## 4. Наложение PNG

Альфа-канал определяет прозрачность. Координаты x, y задают левый верхний угол вставки, ширина - ее размер в пикселях. Вся вставка должна помещаться в исходное изображение

In [ ]:
def overlay_png(base, overlay, x, y, overlay_width):
    if overlay.ndim != 3 or overlay.shape[2] != 4 or overlay.dtype != np.uint8:
        raise ValueError("Нужен PNG с четырьмя каналами BGRA, 8 бит на канал")
    if overlay_width < 1:
        raise ValueError("Ширина должна быть положительной")
    overlay_height = max(1, round(overlay.shape[0] * overlay_width / overlay.shape[1]))
    resized = cv2.resize(overlay, (overlay_width, overlay_height))
    h, w = base.shape[:2]
    if x < 0 or y < 0 or x + overlay_width > w or y + overlay_height > h:
        raise ValueError("Вставка выходит за границы изображения. Уменьшите размер или измените x, y")
    result = base.copy()
    region = result[y:y + overlay_height, x:x + overlay_width].astype(np.float32)
    alpha = resized[:, :, 3:4].astype(np.float32) / 255.0
    mixed = alpha * resized[:, :, :3].astype(np.float32) + (1 - alpha) * region
    result[y:y + overlay_height, x:x + overlay_width] = np.clip(mixed, 0, 255).astype(np.uint8)
    return result


base = read_image(BASE_PATH)
base_height, base_width_pixels = base.shape[:2]
layer = read_image(OVERLAY_PATH, unchanged=True)
# Два начальных варианта. Координаты и размеры можно изменить
base_width = max(1, int(min(base_width_pixels * 0.25, base_height * 0.4 * layer.shape[1] / layer.shape[0])))
PLACEMENTS = [(0, 0, base_width), (base_width_pixels // 2, base_height // 2, base_width)]
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, (x, y, overlay_width) in zip(axes, PLACEMENTS):
    combined = overlay_png(base, layer, x, y, overlay_width)
    show_bgr(ax, combined, f"x={x}, y={y}, ширина={overlay_width}")
plt.show()

Наблюдения: расположение вставки, прозрачные и полупрозрачные участки, результат изменения размера

## 5. Подбор цвета фона

HSV разделяет цветовой тон H, насыщенность S и яркость V. Для 8-битного изображения в OpenCV H принимает значения от 0 до 179, S и V - от 0 до 255

Маска равна 255 в пикселях, попавших в диапазон. Эти пиксели заменяются новым фоном. Два начальных диапазона ниже предназначены для зеленого фона и требуют проверки на собственном видео

In [ ]:
def read_video_frame(path, frame_index):
    if frame_index < 0:
        raise ValueError("Номер кадра не может быть отрицательным")
    capture = cv2.VideoCapture(str(path))
    try:
        if not capture.isOpened():
            raise ValueError(f"Не удалось открыть видео: {path}")
        fps = capture.get(cv2.CAP_PROP_FPS)
        capture.set(cv2.CAP_PROP_POS_FRAMES, frame_index)
        ok, frame = capture.read()
        if not ok:
            raise ValueError(f"Не удалось прочитать кадр {frame_index}")
        return frame, fps
    finally:
        capture.release()


def fit_background(background, shape):
    target_h, target_w = shape[:2]
    scale = max(target_w / background.shape[1], target_h / background.shape[0])
    new_w = max(target_w, int(np.ceil(background.shape[1] * scale)))
    new_h = max(target_h, int(np.ceil(background.shape[0] * scale)))
    resized = cv2.resize(background, (new_w, new_h))
    x = (new_w - target_w) // 2
    y = (new_h - target_h) // 2
    return resized[y:y + target_h, x:x + target_w]


def replace_background(frame, fitted_background, lower, upper):
    lower = np.asarray(lower)
    upper = np.asarray(upper)
    if (lower.shape != (3,) or upper.shape != (3,) or
            np.any(lower < 0) or np.any(upper > [179, 255, 255]) or np.any(lower > upper)):
        raise ValueError("Некорректные границы HSV")
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, lower.astype(np.uint8), upper.astype(np.uint8))
    result = np.where(mask[:, :, None] == 255, fitted_background, frame)
    return result, mask


frame, fps = read_video_frame(VIDEO_PATH, FRAME_INDEX)
if not np.isfinite(fps) or fps <= 0:
    raise ValueError("В видео не указана корректная частота кадров")
background = read_image(BACKGROUND_PATH)
fitted_background = fit_background(background, frame.shape)
print("Размер кадра:", frame.shape, "Частота кадров:", fps, "кадр/с")
print("Время выбранного кадра для видео с постоянным FPS:", FRAME_INDEX / fps, "с")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
show_bgr(axes[0], frame, "Исходный кадр")
show_bgr(axes[1], fitted_background, "Новый фон после подгонки размера")
plt.show()

In [ ]:
HSV_RANGES = [
    ((35, 60, 50), (85, 255, 255)),
    ((45, 15, 20), (95, 255, 255)),
]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for row, (lower, upper) in zip(axes, HSV_RANGES):
    result, mask = replace_background(frame, fitted_background, lower, upper)
    show_bgr(row[0], frame, "Исходный кадр")
    row[1].imshow(mask, cmap="gray", vmin=0, vmax=255)
    row[1].set_title(f"Маска {lower} - {upper}")
    row[1].set_axis_off()
    show_bgr(row[2], result, "Результат")
plt.show()

Выбранный диапазон и причина выбора:

Дефекты по краям объекта, оставшиеся участки фона, ошибочно замененные части объекта:

## 6. Обработка видео

Все кадры обрабатываются выбранным диапазоном. Результат сохраняется в outputs/result.avi с кодеком MJPG и исходным FPS. Звуковая дорожка не переносится

Для этого шаблона нужно видео с постоянной частотой кадров. AVI может не воспроизводиться внутри браузера, поэтому ниже проверяются кадры из записанного файла

In [ ]:
SELECTED_RANGE = 0  # Индекс выбранного диапазона: 0 или 1
RESULT_PATH = OUTPUT_DIR / "result.avi"


def process_video(source, destination, background, lower, upper):
    if Path(source).resolve() == Path(destination).resolve():
        raise ValueError("Исходный файл и результат должны иметь разные пути")
    capture = cv2.VideoCapture(str(source))
    writer = None
    count = 0
    try:
        if not capture.isOpened():
            raise ValueError(f"Не удалось открыть видео: {source}")
        fps = capture.get(cv2.CAP_PROP_FPS)
        if not np.isfinite(fps) or fps <= 0:
            raise ValueError("Некорректный FPS")
        declared_frames = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
        ok, frame = capture.read()
        if not ok:
            raise ValueError("Видео не содержит читаемых кадров")
        h, w = frame.shape[:2]
        fitted = fit_background(background, frame.shape)
        writer = cv2.VideoWriter(str(destination), cv2.VideoWriter_fourcc(*"MJPG"), fps, (w, h))
        if not writer.isOpened():
            raise RuntimeError("Не удалось создать видео с кодеком MJPG")
        while ok:
            if frame.shape[:2] != (h, w):
                raise ValueError("Размер кадра изменился внутри видео")
            result, _ = replace_background(frame, fitted, lower, upper)
            writer.write(result)
            count += 1
            ok, frame = capture.read()
    finally:
        capture.release()
        if writer is not None:
            writer.release()
    if declared_frames > 0 and count != declared_frames:
        raise RuntimeError(f"Прочитано {count} кадров, в метаданных указано {declared_frames}. Нужно проверить исходный ролик")
    return count, fps, (w, h)


lower, upper = HSV_RANGES[SELECTED_RANGE]
written_count, source_fps, source_size = process_video(
    VIDEO_PATH, RESULT_PATH, background, lower, upper,
)
print("Обработано кадров:", written_count)
print("Результат:", RESULT_PATH)

In [ ]:
# Повторное чтение проверяет, что файл содержит ожидаемое число кадров и размер
capture = cv2.VideoCapture(str(RESULT_PATH))
preview_frames = []
preview_indices = sorted({0, written_count // 2, written_count - 1})
read_count = 0
try:
    if not capture.isOpened():
        raise RuntimeError("Не удалось открыть записанный результат")
    result_fps = capture.get(cv2.CAP_PROP_FPS)
    while True:
        ok, result_frame = capture.read()
        if not ok:
            break
        if (result_frame.shape[1], result_frame.shape[0]) != source_size:
            raise RuntimeError("Размер записанного кадра отличается от исходного")
        if read_count in preview_indices:
            preview_frames.append((read_count, result_frame))
        read_count += 1
finally:
    capture.release()
if read_count != written_count or not np.isclose(result_fps, source_fps, rtol=0.001):
    raise RuntimeError("Число кадров или FPS результата отличается от ожидаемого")
print("Прочитано кадров:", read_count, "FPS:", result_fps, "Размер:", source_size)
fig, axes = plt.subplots(1, len(preview_frames), figsize=(15, 5), squeeze=False)
for ax, (index, preview) in zip(axes[0], preview_frames):
    show_bgr(ax, preview, f"Кадр {index}, время {index / result_fps:.2f} с")
plt.show()

## Выводы

Что удалось получить в каждой задаче:

Влияние параметров детектора лиц и диапазона HSV:

Ограничения результата и возможные улучшения:

Для сдачи: выполненный ipynb с результатами и выводами, обработанный видеофайл. В Colab файлы из outputs скачиваются до завершения сеанса